<p align="center" style="text-align:center; margin:4px 0; font-size:24px; line-height:1.3;"><strong>INSTITUTO INTERNACIONAL DE AGUASCALIENTES</strong></p>
<p align="center" style="text-align:center; margin:4px 0; font-size:20px; line-height:1.3;"><strong>Maestría en Inteligencia Artificial para la Transformación Digital</strong></p>
<p align="center" style="text-align:center; margin:4px 0; font-size:18px; line-height:1.3;">Programa: Optimización</p>
<p align="center" style="text-align:center; margin:8px 0; font-size:26px; line-height:1.3;"><strong>Unidad 1 - Práctica: Solución de Ejercicios de Optimización Numérica</strong></p>
<p align="center" style="text-align:center; margin:4px 0; font-size:16px; line-height:1.3;">Alumno: <strong>Antonio Nicolás Toro González</strong></p>
<p align="center" style="text-align:center; margin:4px 0; font-size:16px; line-height:1.3;">Tutor: <strong>Dr. Luis Julián Domínguez Pérez</strong></p>
<p align="center" style="text-align:center; margin:4px 0; font-size:16px; line-height:1.3;">noviembre de 2026</p>
<hr>

# Práctica: Solución de Ejercicios | Optimización numérica

**Manufactura de muebles · Python / Google Colab**

## 1. Introducción y planteamiento

La optimización numérica permite determinar los valores de las variables de decisión que maximizan o minimizan una función objetivo.

En esta práctica se desarrollarán los ejercicios presentados por el Dr. Luis Julián Domínguez Pérez (2026), con el propósito de determinar la cantidad de muebles que maximiza las ganancias y analizar cómo el capital y el trabajo influyen en la producción. Para ello, se aplicarán la búsqueda exhaustiva, la primera y segunda derivada y las derivadas parciales mediante Python en Google Colab.

Como complemento, se planteará una propuesta alternativa para identificar las combinaciones de horas-máquina y horas-hombre que permitan alcanzar la producción óptima y la ganancia máxima. Cada procedimiento incluirá su fundamentación matemática, implementación computacional, representación gráfica y conclusión de resultados.

### 1.1. Problema univariado: maximización de ganancias

Determinar la cantidad de muebles que maximiza la ganancia.

$$\boxed{P(q)=40q-0.5q^2}$$

Donde:

- $P(q)$: ganancia obtenida.
- $q$: cantidad de muebles producidos.

### 1.2. Problema multivariado: función de producción

Analizar la producción en función del capital y el trabajo.

$$\boxed{F(K,L)=K^{0.34}L^{0.66}}$$

Donde:

- $F(K,L)$: producción obtenida.
- $K$: capital (horas-máquina).
- $L$: trabajo (horas-hombre).

### 1.3. Propuesta alternativa

Tomar como referencia los resultados de producción óptima y ganancia máxima del problema univariado para determinar las posibles combinaciones de capital y trabajo que permitan alcanzarlos.

## 2. Bibliotecas y función objetivo

| Biblioteca | Aplicación en la práctica |
|---|---|
| NumPy | Cálculo numérico y búsqueda exhaustiva. |
| SymPy | Cálculo de derivadas y resolución de ecuaciones. |
| Matplotlib | Representación gráfica bidimensional y tridimensional. |

In [ ]:
import logging
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math
from matplotlib.axes import Axes
from matplotlib_inline.backend_inline import set_matplotlib_formats, InlineBackend

# Mostrar las figuras completas en el notebook, con margen alrededor.
set_matplotlib_formats('png')
InlineBackend.instance().print_figure_kwargs = {'bbox_inches': None}
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlepad': 12})

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("optimizacion_practica_1")


def ganancia(q: np.ndarray | float) -> np.ndarray:
    """Evalúa P(q) para cantidades reales, finitas y no negativas."""
    valores = np.asarray(q, dtype=float)
    if valores.size == 0 or not np.all(np.isfinite(valores)) or np.any(valores < 0):
        raise ValueError("Las cantidades deben ser finitas y no negativas.")
    with np.errstate(over="raise", invalid="raise"):
        return 40 * valores - 0.5 * valores**2


def mostrar_ecuacion(expresion: sp.Basic, etiqueta: str) -> None:
    """Presenta una expresión simbólica con notación matemática."""
    display(Math(etiqueta + "=" + sp.latex(expresion)))


def configurar_ejes(ax: Axes, titulo: str, x: str, y: str) -> None:
    """Configura etiquetas y cuadrícula de una gráfica bidimensional."""
    ax.set(title=titulo, xlabel=x, ylabel=y)
    ax.grid(alpha=0.25)


q = sp.symbols("q", real=True)
P = 40*q - sp.Rational(1, 2)*q**2
cantidades = np.arange(80)
malla_q = np.linspace(0, 79, 400)
print("Bibliotecas listas. Se evaluarán 80 cantidades enteras (0 a 79).")

### Representación inicial de la función objetivo

Se representa gráficamente la relación entre la cantidad de muebles producidos y la ganancia obtenida.

$$P(q)=40q-0.5q^2$$

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8), layout="constrained")
ax.plot(malla_q, ganancia(malla_q), label="P(q)", color="tab:blue")
configurar_ejes(ax, "Función objetivo de ganancias", "Cantidad de muebles (q)", "Ganancia P(q)")
ax.legend()
plt.show()
plt.close(fig)

**Interpretación:** la función presenta una forma parabólica en la que las ganancias aumentan hasta alcanzar un punto máximo y posteriormente disminuyen.

## 3. Método 1. Búsqueda exhaustiva (fuerza bruta)

Consiste en evaluar todas las alternativas de un conjunto finito y seleccionar aquella que proporciona el mayor valor de la función objetivo.

**Fundamento matemático:**

$$x^*=\arg\max_{x\in S}f(x)$$

**Aplicación al problema:**

$$q^*=\arg\max_{q\in S}(40q-0.5q^2)$$

$$S=\{0,1,\ldots,79\}$$

In [ ]:
def busqueda_exhaustiva(candidatos: np.ndarray) -> tuple[int, float, np.ndarray]:
    """Devuelve la mejor cantidad, su ganancia y las ganancias evaluadas."""
    valores = np.asarray(candidatos)
    if valores.ndim != 1 or valores.size == 0:
        raise ValueError("Se requiere un vector no vacío.")
    if not np.issubdtype(valores.dtype, np.integer) or np.any(valores < 0):
        raise ValueError("Se requieren cantidades enteras no negativas.")
    beneficios = ganancia(valores)
    indice = int(np.argmax(beneficios))
    return int(valores[indice]), float(beneficios[indice]), beneficios


try:
    q_bruta, p_bruta, beneficios = busqueda_exhaustiva(cantidades)
except (ValueError, TypeError, FloatingPointError) as error:
    logger.error("Error en búsqueda exhaustiva: %s", error)
    raise

logger.info("Búsqueda exhaustiva completada.")
print(f"Valores evaluados: {cantidades.size}")
print(f"Cantidad óptima: {q_bruta} muebles")
print(f"Ganancia máxima: {p_bruta:.2f}")

fig, ax = plt.subplots(figsize=(8.5, 4.8), layout="constrained")
ax.plot(cantidades, beneficios, "o-", markersize=3, color="tab:blue", label="Cantidades evaluadas")
ax.scatter(q_bruta, p_bruta, color="tab:red", s=85, zorder=5,
           label=f"Máximo ({q_bruta}, {p_bruta:g})")
ax.axvline(q_bruta, color="tab:red", linestyle="--", alpha=0.5)
configurar_ejes(ax, "Búsqueda exhaustiva", "Cantidad de muebles (q)", "Ganancia P(q)")
ax.legend()
plt.show()
plt.close(fig)

**Conclusión del método 1:** de las 80 cantidades evaluadas, la producción de 40 muebles proporciona la mayor ganancia, equivalente a 800 unidades monetarias.

## 4. Método 2. Primera derivada

La primera derivada representa la tasa de variación de una función y permite identificar sus puntos estacionarios.

**Fundamento matemático:**

$$f'(x)=\frac{df}{dx}=0$$

**Aplicación al problema:**

$$P'(q)=\frac{d}{dq}(40q-0.5q^2)$$

In [ ]:
def punto_estacionario(funcion: sp.Expr, variable: sp.Symbol) -> tuple[sp.Expr, sp.Expr]:
    """Calcula la primera derivada y el único candidato real no negativo."""
    derivada = sp.diff(funcion, variable)
    candidatos = [
        valor for valor in sp.solve(derivada, variable)
        if valor.is_real is True and valor.is_nonnegative is True
    ]
    if len(candidatos) != 1:
        raise ValueError("Se esperaba un único punto estacionario admisible.")
    return derivada, candidatos[0]


try:
    dP, q_est = punto_estacionario(P, q)
except ValueError as error:
    logger.error("No se pudo identificar el candidato: %s", error)
    raise

p_est = P.subs(q, q_est)
mostrar_ecuacion(dP, r"P'(q)")
print(f"Punto estacionario: q = {q_est}")
print(f"Ganancia en el punto estacionario: {p_est}")
logger.info("Primera derivada calculada.")

pendiente = sp.lambdify(q, dP, "numpy")
fig, ax = plt.subplots(figsize=(8.5, 4.8), layout="constrained")
ax.plot(malla_q, pendiente(malla_q), color="tab:blue", label="P'(q)")
ax.axhline(0, color="gray", linewidth=1)
ax.scatter(float(q_est), 0, color="tab:red", s=80, zorder=5,
           label=f"Punto estacionario ({q_est}, 0)")
configurar_ejes(ax, "Primera derivada de la ganancia", "Cantidad de muebles (q)", "Pendiente P'(q)")
ax.legend()
plt.show()
plt.close(fig)

**Conclusión del método 2:** al igualar la primera derivada a cero, se identificó que la cantidad óptima de producción es de 40 muebles. Al sustituir este valor en la función objetivo, se obtuvo una ganancia de 800 unidades monetarias. La naturaleza del punto estacionario se comprobará mediante la segunda derivada en el siguiente método.

## 5. Método 3. Segunda derivada

La segunda derivada permite analizar la curvatura de una función y clasificar sus puntos estacionarios.

**Fundamento matemático:**

$$f''(x)=\frac{d^2f}{dx^2}$$

| Condición en un punto estacionario | Interpretación |
|---|---|
| $f''(x)<0$ | Máximo local. |
| $f''(x)>0$ | Mínimo local. |
| $f''(x)=0$ | Resultado no concluyente. |

**Aplicación al problema:**

$$P''(q)=\frac{d^2}{dq^2}(40q-0.5q^2)$$

In [ ]:
def clasificar_estacionario(funcion: sp.Expr, variable: sp.Symbol,
                            candidato: sp.Expr) -> tuple[sp.Expr, sp.Expr, str]:
    """Evalúa la segunda derivada de un punto estacionario real."""
    if sp.diff(funcion, variable).subs(variable, candidato) != 0:
        raise ValueError("El candidato debe ser estacionario.")
    segunda = sp.diff(funcion, variable, 2)
    valor = segunda.subs(variable, candidato)
    if valor.is_negative is True:
        tipo = "Máximo local"
    elif valor.is_positive is True:
        tipo = "Mínimo local"
    else:
        tipo = "Criterio no concluyente"
    return segunda, valor, tipo


d2P, curvatura, clasificacion = clasificar_estacionario(P, q, q_est)
mostrar_ecuacion(d2P, r"P''(q)")
print(f"Segunda derivada en q = {q_est}: {curvatura}")
print(f"Clasificación: {clasificacion}")
if d2P.is_negative is True:
    print("Concavidad estricta en todo el dominio: máximo global único.")
logger.info("Clasificación mediante segunda derivada completada.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 7.5), sharex=True, layout="constrained")
ax1.plot(malla_q, ganancia(malla_q), color="tab:blue")
ax1.scatter(float(q_est), float(p_est), color="tab:red", s=80, zorder=5,
            label=f"Máximo ({q_est}, {p_est})")
configurar_ejes(ax1, "Función original", "", "Ganancia P(q)")
ax1.legend()

curva_segunda = sp.lambdify(q, d2P, "numpy")
valores_segunda = np.broadcast_to(curva_segunda(malla_q), malla_q.shape)
ax2.plot(malla_q, valores_segunda, color="tab:purple", label="P''(q)")
ax2.axhline(0, color="gray", linewidth=1)
ax2.set_ylim(-1.5, 0.5)
configurar_ejes(ax2, "Segunda derivada", "Cantidad de muebles (q)", "Curvatura P''(q)")
ax2.legend()
plt.show()
plt.close(fig)

**Conclusión del método 3:** la segunda derivada es negativa y constante, lo que demuestra que la función es estrictamente cóncava. Se confirma que producir 40 muebles genera una ganancia máxima global de 800 unidades monetarias.

## 6. Método 4. Derivadas parciales (multivariado)

Las derivadas parciales permiten analizar la variación de una función respecto de cada variable, manteniendo constantes las demás.

**Fundamento matemático:**

$$\frac{\partial f}{\partial x},\qquad \frac{\partial f}{\partial y}$$

Condición de estacionariedad interior:

$$\frac{\partial f}{\partial x}=0,\qquad \frac{\partial f}{\partial y}=0$$

**Aplicación al problema:**

$$F(K,L)=K^{0.34}L^{0.66}$$

$$\frac{\partial F}{\partial K},\qquad \frac{\partial F}{\partial L}$$

El análisis mediante derivadas parciales se realiza para valores positivos de capital y trabajo ($K,L>0$), mientras que la representación gráfica incluye el cero, al estar definida la función de producción para $K,L\geq0$.

In [ ]:
def produccion(capital: np.ndarray, trabajo: np.ndarray) -> np.ndarray:
    """Evalúa F(K,L) para entradas finitas, compatibles y no negativas."""
    k, l = np.broadcast_arrays(np.asarray(capital, dtype=float),
                               np.asarray(trabajo, dtype=float))
    if k.size == 0 or not np.all(np.isfinite(k)) or not np.all(np.isfinite(l)):
        raise ValueError("Capital y trabajo deben contener valores finitos.")
    if np.any(k < 0) or np.any(l < 0):
        raise ValueError("Capital y trabajo no pueden ser negativos.")
    with np.errstate(over="raise", invalid="raise"):
        return k**0.34 * l**0.66


def analizar_parciales(funcion: sp.Expr, k: sp.Symbol, l: sp.Symbol) -> tuple[sp.Expr, sp.Expr, list[dict]]:
    """Devuelve derivadas parciales y candidatos estacionarios interiores."""
    parcial_k = sp.diff(funcion, k)
    parcial_l = sp.diff(funcion, l)
    candidatos = sp.solve([parcial_k, parcial_l], (k, l), dict=True)
    return parcial_k, parcial_l, candidatos


K, L = sp.symbols("K L", positive=True)
F = K**sp.Rational(17, 50) * L**sp.Rational(33, 50)
dF_K, dF_L, criticos = analizar_parciales(F, K, L)

mostrar_ecuacion(dF_K, r"\frac{\partial F}{\partial K}")
mostrar_ecuacion(dF_L, r"\frac{\partial F}{\partial L}")
print(f"Puntos estacionarios interiores: {criticos}")
print(f"Ambas derivadas son positivas: {dF_K.is_positive and dF_L.is_positive}")

t = sp.symbols("t", positive=True)
diagonal = sp.simplify(F.subs({K: t, L: t}))
mostrar_ecuacion(diagonal, "F(t,t)")
print("No existe un máximo finito sin restricciones de recursos.")
logger.info("Análisis multivariado completado.")

# Rango de visualización del ejemplo del profesor: 0 a 9.75.
rango = np.arange(0, 10, 0.25)
X, Y = np.meshgrid(rango, rango)
Z = produccion(X, Y)

fig = plt.figure(figsize=(9, 6.6))
ax = fig.add_axes([0.15, 0.12, 0.64, 0.76], projection="3d")
cax = fig.add_axes([0.86, 0.25, 0.022, 0.48])
superficie = ax.plot_surface(X, Y, Z, cmap="Blues", edgecolor="none",
                              antialiased=True)
ax.set(title="Superficie de producción", xlabel="K (horas-máquina)",
       ylabel="L (horas-hombre)", zlabel="Producción F(K,L)")
ax.view_init(elev=28, azim=-130)
ax.set_xlabel("K (horas-máquina)", labelpad=12)
ax.set_ylabel("L (horas-hombre)", labelpad=12)
ax.set_zlabel("Producción F(K,L)", labelpad=12)
fig.colorbar(superficie, cax=cax, label="Producción")
plt.show()
plt.close(fig)

**Conclusión del método 4:** la producción aumenta al incrementar las horas-máquina o las horas-hombre, manteniendo el otro recurso en un valor positivo. No se obtiene una producción máxima porque el problema no establece límites para los recursos.

### 6.1. Solución alternativa: optimización multivariada a partir de la producción y la ganancia óptimas

Como propuesta de elaboración propia, se toman en consideración los resultados obtenidos anteriormente:

$$q^*=40,\qquad P_{\max}=800$$

Se establece como supuesto que la función de producción representa la cantidad de muebles fabricados:

$$q=F(K,L)=K^{0.34}L^{0.66}$$

**Función objetivo multivariada:**

$$P(K,L)=40F(K,L)-0.5[F(K,L)]^2$$

**Derivadas parciales mediante la regla de la cadena:**

$$\frac{\partial P}{\partial K}=(40-F)\,0.34K^{-0.66}L^{0.66}$$

$$\frac{\partial P}{\partial L}=(40-F)\,0.66K^{0.34}L^{-0.34}$$

**Condiciones de estacionariedad:**

$$\frac{\partial P}{\partial K}=0,\qquad \frac{\partial P}{\partial L}=0$$

Para $K,L>0$, las derivadas parciales de $F$ son positivas. Por tanto, ambas derivadas de $P$ se anulan cuando $40-F(K,L)=0$.

**Condición de producción óptima:**

$$K^{0.34}L^{0.66}=40$$

**Combinaciones de capital y trabajo:**

$$L=\left(\frac{40}{K^{0.34}}\right)^{1/0.66}$$

**Comprobación del máximo global:**

$$P(K,L)=800-0.5[F(K,L)-40]^2$$

$$F(K,L)=40\implies P_{\max}=800$$


In [ ]:
def trabajo_optimo(capital: np.ndarray | float, objetivo: float) -> np.ndarray:
    """Calcula las horas-hombre para alcanzar la producción objetivo."""
    k = np.asarray(capital, dtype=float)
    if k.size == 0 or not np.all(np.isfinite(k)) or np.any(k <= 0):
        raise ValueError("El capital debe ser finito y estrictamente positivo.")
    if not np.isfinite(objetivo) or objetivo <= 0:
        raise ValueError("La producción objetivo debe ser finita y positiva.")
    with np.errstate(over="raise", divide="raise", invalid="raise"):
        l = (objetivo / k**0.34)**(1 / 0.66)
    if not np.all(np.isfinite(l)) or np.any(l <= 0):
        raise ValueError("El trabajo calculado debe ser finito y positivo.")
    return l


try:
    # Reutilizar los resultados de los métodos anteriores.
    objetivo = float(q_est)
    maximo = float(p_est)
    if not np.isclose(q_bruta, objetivo) or not np.isclose(p_bruta, maximo) or d2P.is_negative is not True:
        raise ValueError("Los métodos anteriores no confirman el mismo máximo.")

    P_integrada = P.subs(q, F)
    dP_K = sp.factor(sp.diff(P_integrada, K))
    dP_L = sp.factor(sp.diff(P_integrada, L))
    curva_exacta = (q_est / K**sp.Rational(17, 50))**sp.Rational(50, 33)
    mostrar_ecuacion(P_integrada, r"P(K,L)")
    mostrar_ecuacion(dP_K, r"\frac{\partial P}{\partial K}")
    mostrar_ecuacion(dP_L, r"\frac{\partial P}{\partial L}")
    mostrar_ecuacion(curva_exacta, r"L^*(K)")

    # Valores ilustrativos, no límites de disponibilidad.
    k_ejemplos = objetivo * np.array([0.25, 0.5, 1, 1.5, 2, 2.5])
    l_ejemplos = trabajo_optimo(k_ejemplos, objetivo)
    q_ejemplos = produccion(k_ejemplos, l_ejemplos)
    p_ejemplos = ganancia(q_ejemplos)

    print("COMBINACIONES ÓPTIMAS ILUSTRATIVAS")
    print(f"{'Horas-máquina':>15} {'Horas-hombre':>15} {'Muebles':>12} {'Ganancia':>12}")
    for k_val, l_val, q_val, p_val in zip(k_ejemplos, l_ejemplos, q_ejemplos, p_ejemplos):
        print(f"{k_val:15.2f} {l_val:15.2f} {q_val:12.2f} {p_val:12.2f}")

    # Validación simbólica y numérica de la solución alternativa.
    assert sp.simplify(P_integrada - (p_est + d2P / 2 * (F - q_est)**2)) == 0
    assert sp.simplify(F.subs(L, curva_exacta) - q_est) == 0
    assert sp.simplify(dP_K.subs(L, curva_exacta)) == 0
    assert sp.simplify(dP_L.subs(L, curva_exacta)) == 0
    assert np.allclose(q_ejemplos, objetivo)
    assert np.allclose(p_ejemplos, maximo)
    for k_invalido in (0, -1, np.nan):
        try:
            trabajo_optimo(k_invalido, objetivo)
        except ValueError:
            continue
        raise AssertionError("No se rechazó un capital inválido.")
    logger.info("Propuesta alternativa comprobada.")
except (ValueError, TypeError, FloatingPointError, AssertionError) as error:
    logger.error("Error en la propuesta alternativa: %s", error)
    raise

In [ ]:
# Isocuanta: combinaciones de recursos con producción óptima.
k_curva = np.linspace(k_ejemplos.min(), k_ejemplos.max(), 400)
l_curva = trabajo_optimo(k_curva, objetivo)
fig, ax = plt.subplots(figsize=(7.2, 4.0), dpi=100, layout="constrained")
ax.plot(k_curva, l_curva, label=f"F(K,L) = {objetivo:g}", color="tab:blue")
ax.scatter(k_ejemplos, l_ejemplos, color="tab:red", s=35,
           label="Combinaciones ilustrativas", zorder=3)
configurar_ejes(ax, "Isocuanta de producción óptima",
                "K (horas-máquina)", "L (horas-hombre)")
ax.legend()
plt.show()
plt.close(fig)


In [ ]:
# Superficie de ganancias y curva de máximos.
k_malla, l_malla = np.meshgrid(
    np.linspace(0, k_curva.max() * 1.1, 140),
    np.linspace(0, l_curva.max() * 1.1, 140))
p_malla = ganancia(produccion(k_malla, l_malla))
fig = plt.figure(figsize=(7.6, 5.3), dpi=100)
ax = fig.add_axes([0.14, 0.14, 0.62, 0.70], projection="3d")
cax = fig.add_axes([0.83, 0.27, 0.022, 0.43])
superficie_p = ax.plot_surface(k_malla, l_malla, p_malla,
                               cmap="viridis", alpha=0.85, edgecolor="none")
ax.plot(k_curva, l_curva, np.full_like(k_curva, maximo),
        color="tab:red", linewidth=3, label=f"Máximos: P = {maximo:g}", zorder=10)
ax.set(title="Superficie multivariada de ganancias",
       xlabel="K (horas-máquina)", ylabel="L (horas-hombre)", zlabel="Ganancia P(K,L)")
ax.view_init(elev=30, azim=-130)
ax.set_xlabel("K (horas-máquina)", labelpad=7)
ax.set_ylabel("L (horas-hombre)", labelpad=7)
ax.set_zlabel("Ganancia P(K,L)", labelpad=7)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.01))
fig.colorbar(superficie_p, cax=cax, label="Ganancia")
plt.show()
plt.close(fig)


**Conclusión de la propuesta alternativa:** existen múltiples combinaciones de horas-máquina y horas-hombre que permiten producir 40 muebles y alcanzar una ganancia máxima de 800 unidades monetarias, bajo el supuesto de producción planteado. Para seleccionar una combinación específica sería necesario conocer los costos o la disponibilidad de los recursos y establecer un criterio de selección.

*Elaboración propia a partir de las funciones presentadas por el Dr. Domínguez Pérez (2026).*

## 7. Prospección: análisis de sensibilidad de las ganancias

Se propone explorar cómo varían las ganancias ante cambios en la cantidad de muebles producidos respecto al nivel óptimo identificado.

Partiendo de la función objetivo:

$$P(q)=40q-0.5q^2$$

Se analizarían diferentes escenarios de producción para responder las siguientes preguntas:

- ¿Qué impacto económico tendría producir por encima o por debajo del nivel óptimo?
- ¿Qué margen de variación permitiría mantener una ganancia aceptable?
- ¿Cómo podrían aprovecharse estos resultados para mejorar la planificación de la producción?

**Propósito:** explorar la sensibilidad de las ganancias ante variaciones en la producción y evaluar su utilidad para la toma de decisiones, sin anticipar conclusiones antes de realizar la prueba. Para determinar un margen aceptable deberá definirse previamente un umbral de ganancia.

Esta propuesta queda pendiente de ejecución y análisis.


## 8. Resultados y conclusión general

| Método | Resultado |
|---|---|
| Búsqueda exhaustiva | Producción óptima de 40 muebles y ganancia de 800. |
| Primera derivada | Punto estacionario en $q=40$. |
| Segunda derivada | Confirmación del máximo global. |
| Derivadas parciales | Producción creciente sin máximo finito. |
| Propuesta alternativa | Múltiples combinaciones de recursos para alcanzar la producción y ganancia óptimas. |

**Conclusión general:** los ejercicios permitieron determinar la cantidad de producción que maximiza las ganancias y analizar el comportamiento de los factores productivos. La propuesta alternativa permitió establecer posibles combinaciones de capital y trabajo para alcanzar la producción y la ganancia óptimas, bajo el supuesto planteado.

### 8.1. Reflexión sobre lo aprendido

La práctica permitió comprender la relación entre la formulación matemática, los métodos de optimización y su implementación computacional. También permitió reconocer la importancia de interpretar los resultados, identificar las limitaciones de los modelos y aprovechar el conocimiento obtenido para plantear nuevas alternativas de solución.

## **9. Referencias de consulta**

1. Domínguez Pérez, L. J. (2026). *Optimización numérica: Introducción* [Diapositivas de presentación]. Instituto Internacional de Aguascalientes.

2. Maciel, M. C. (2000). *Notas de optimización: Introducción a la optimización numérica* [Notas de clase / Manuscrito no publicado]. Facultad de Matemática, Astronomía y Física, Universidad Nacional de Córdoba.

3. Matplotlib Development Team. (2026). *Matplotlib documentation*. [https://matplotlib.org/stable/](https://matplotlib.org/stable/)

4. NumPy Developers. (2026). *NumPy documentation*. [https://numpy.org/doc/stable/](https://numpy.org/doc/stable/)

5. SymPy Development Team. (2026). *SymPy documentation*. [https://docs.sympy.org/latest/](https://docs.sympy.org/latest/)